# 실습 - 이미지 분류

**철도 궤도 결함 판별**

**작업 개요**

- 작업 이름 — `image-classification`
- 입력 — 이미지 한 장
- 출력 — 라벨(label)과 점수(score)
- 판별 범위 — 모델이 학습한 분류 체계 내

**데이터**

- 궤도 표면을 촬영한 점검용 이미지
- 범주 2종 — `Defective`(결함) · `Non-defective`(정상)
- 정답 라벨 포함 — 예측과 대조 가능

**모델**

- 동일 데이터로 학습된 ViT 분류 모델

**실습 순서**

- 환경 준비
- 데이터 불러오기 · 라벨 확인
- 모델 준비
- 추론 실행 · 결과 읽기
- 정확도 측정

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행
- `torch` — GPU 사용 여부 확인

**경고 메시지 안내**

- 실행 시 붉은 경고 문구 출력 가능
- 경고(Warning)와 오류(Error)는 구분 — 결과가 출력되면 정상

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import torch

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- `load_dataset` — Hub 데이터셋 내려받기
- `split="train[:100]"` — 전체가 아닌 앞 100장만 사용
- 첫 실행은 내려받기로 시간 소요 — 이후 저장된 파일 재사용

In [ ]:
ds = load_dataset(
    "crangana/railroad-fault-detection",
    split="train[:100]"
)

ds

### 1-2. 라벨 확인

- `features["label"].names` — 범주 이름 목록
- 정답은 **번호**로 저장 — 이름 목록과 대조 필요
- 이후 예측 결과 해석의 기준

In [ ]:
labels = ds.features["label"].names

print("범주 수:", len(labels))
print("범주 이름:", labels)
print("무작위 확률:", round(1 / len(labels), 2))

### 1-3. 이미지와 정답 확인

- `ds[0]["image"]` — 첫 번째 이미지
- `ds[0]["label"]` — 정답 번호
- 이미지와 정답을 눈으로 대조

In [ ]:
image = ds[0]["image"]

print("정답:", labels[ds[0]["label"]])
image

### 1-4. 데이터 구성 확인

- 결함 · 정상이 각각 몇 장인지 확인
- 한쪽으로 치우치면 정확도 해석에 주의 필요

In [ ]:
from collections import Counter

counts = Counter(ds["label"])
for i, n in sorted(counts.items()):
    print(labels[i], "-", n, "장")

---
## 2. 모델 준비

- `task` — 수행할 작업 이름
- `model` — 궤도 데이터로 학습된 모델
- `device=0` — GPU에서 실행 (미지정 시 CPU, 처리 속도 저하)

첫 실행 시 모델 내려받기로 시간이 소요된다.

In [ ]:
clf = pipeline(
    task="image-classification",
    model="crangana/railroad-fault-detector",
    device=0
)

print("모델 범주:", clf.model.config.id2label)

---
## 3. 추론 실행

- 이미지를 입력하면 즉시 결과 반환
- 모델 파라미터는 변경되지 않음 — 추론 전용

In [ ]:
result = clf(image)
result

### 3-1. 결과 읽기

- **label** — 모델이 예측한 범주 이름
- **score** — 해당 범주에 대한 예측 확률
- 점수가 높은 순으로 정렬 — 전체 합계 1
- `result[0]` — 최종 예측

In [ ]:
for r in result:
    print(r["label"], "-", round(r["score"], 4))

print()
print("예측:", result[0]["label"])
print("정답:", labels[ds[0]["label"]])

### 3-2. 라벨 표기 확인

- 모델 출력과 데이터 정답의 **표기가 동일해야** 비교 가능
- 대소문자 · 공백이 다르면 정확도 계산이 어긋남

In [ ]:
pred_label = result[0]["label"]
true_label = labels[ds[0]["label"]]

print("모델 출력:", repr(pred_label))
print("데이터 정답:", repr(true_label))
print("표기 일치:", pred_label == true_label)

---
## 4. 정확도 측정

- 100장 전체에 대해 예측과 정답 비교
- 이진 분류이므로 **무작위 확률 50%**와 대조하여 해석
- 소요 시간 — 장당 수십 ms

In [ ]:
import time

t0 = time.time()
correct = 0

for item in ds:
    pred = clf(item["image"])[0]["label"]
    if pred == labels[item["label"]]:
        correct += 1

print("정확도:", round(correct / len(ds), 4))
print("무작위 확률:", round(1 / len(labels), 4))
print("소요 시간:", round(time.time() - t0, 1), "초")

### 4-1. 틀린 사례 확인

- 어떤 이미지에서 틀렸는지 확인
- 결과 해석과 한계 파악에 활용

In [ ]:
wrong = []

for i, item in enumerate(ds):
    pred = clf(item["image"])[0]["label"]
    true = labels[item["label"]]
    if pred != true:
        wrong.append((i, pred, true))

print("틀린 건수:", len(wrong))
for i, p, t in wrong[:5]:
    print(f"  {i}번 — 예측 {p} / 정답 {t}")

In [ ]:
# 틀린 이미지 확인 (틀린 사례가 있는 경우)
if wrong:
    idx = wrong[0][0]
    print("예측:", wrong[0][1], "/ 정답:", wrong[0][2])
    display(ds[idx]["image"])
else:
    print("틀린 사례 없음")

---
## 5. 다른 이미지 확인

- `ds[0]` 의 번호를 바꾸어 다른 이미지 확인
- `top_k` 인자로 출력 후보 수 조절
- 본인이 준비한 이미지로 실행 — `clf("파일경로.jpg")`

In [ ]:
# 다른 이미지 확인
n = 5

print("정답:", labels[ds[n]["label"]])
print("예측:", clf(ds[n]["image"]))
ds[n]["image"]

---
## 6. 정리

**수행 내용**

- 궤도 데이터로 학습된 모델을 그대로 사용 (추론)
- 이미지 한 장 → 라벨과 점수 산출
- 100장에 대해 정확도 측정

**확인한 사항**

- 출력은 모든 범주에 대한 점수 — 합계 1
- 데이터에 맞는 모델을 사용하면 높은 정확도
- 정답 라벨이 있어 성능을 수치로 확인 가능

**주의**

- 모델 출력과 데이터 정답의 표기가 일치해야 비교 가능
- 무작위 확률과 대조하여 성능 해석
- 학습 분류 체계 밖의 대상은 판별 불가

---

데이터·모델 출처: crangana (Hugging Face Hub 공개)